In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics import (accuracy_score,classification_report,confusion_matrix,log_loss)

In [11]:
iris_data_raw = pd.read_csv('iris.csv')
iris_data_raw

,sepal.length,sepal.width,petal.length,petal.width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Virginica
146,6.3,2.5,5.0,1.9,Virginica
147,6.5,3.0,5.2,2.0,Virginica
148,6.2,3.4,5.4,2.3,Virginica


In [12]:
iris_data_raw = iris_data_raw.dropna()
class_mapping = {
    "Setosa": 0,
    "Versicolor": 1,
    "Virginica": 2,
}
iris_data_raw["variety_map"] = iris_data_raw["variety"].map(class_mapping)
X = iris_data_raw.drop(columns={'variety','variety_map'})
features = X.columns.to_list()
y = iris_data_raw['variety_map']
print(pd.Series(y).value_counts(normalize = True),'\n')
iris_data_raw

variety_map
0    0.333333
1    0.333333
2    0.333333
Name: proportion, dtype: float64 



,sepal.length,sepal.width,petal.length,petal.width,variety,variety_map
0,5.1,3.5,1.4,0.2,Setosa,0
1,4.9,3.0,1.4,0.2,Setosa,0
2,4.7,3.2,1.3,0.2,Setosa,0
3,4.6,3.1,1.5,0.2,Setosa,0
4,5.0,3.6,1.4,0.2,Setosa,0
...,...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Virginica,2
146,6.3,2.5,5.0,1.9,Virginica,2
147,6.5,3.0,5.2,2.0,Virginica,2
148,6.2,3.4,5.4,2.3,Virginica,2


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", np.unique(y_train, return_counts=True))
print("y_test:", np.unique(y_test, return_counts=True))

X_train: (120, 4)
X_test: (30, 4)
y_train: (array([0, 1, 2], dtype=int64), array([40, 40, 40], dtype=int64))
y_test: (array([0, 1, 2], dtype=int64), array([10, 10, 10], dtype=int64))


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(8, activation="relu"),
    Dense(3, activation="softmax"),
])

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_2 (Dense)             (None, 8)                 40        
                                                                 
 dense_3 (Dense)             (None, 3)                 27        
                                                                 
Total params: 67 (268.00 Byte)
Trainable params: 67 (268.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [15]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
optimizer = Adam(learning_rate=0.001)
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=100,
    min_lr=0.00001,
    verbose=1,
)


In [16]:
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=2000,
    batch_size=16,
    validation_split=0.2,
    callbacks=[lr_scheduler],
    verbose=0,
)

In [17]:
test_loss, test_accuracy = model.evaluate(
    X_test_scaled,
    y_test,
)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

1/1 [==============================] - 0s 24ms/step - loss: 0.0581 - accuracy: 1.0000
Test loss: 0.058089397847652435
Test accuracy: 1.0


In [18]:
probabilities = model.predict(X_test_scaled)

predictions = probabilities.argmax(axis=1)

print("Prawdopodobieństwa:")
print(probabilities[:10])

print("Przewidziane klasy:")
print(predictions[:10])

1/1 [==============================] - 0s 36ms/step
Prawdopodobieństwa:
[[9.9999988e-01 9.3505314e-08 1.8985669e-20]
 [1.2565101e-06 3.1798062e-01 6.8201816e-01]
 [4.4436422e-03 9.9555570e-01 6.6811987e-07]
 [4.9383449e-04 9.9950576e-01 4.3150729e-07]
 [1.0000000e+00 1.2638022e-08 3.3465581e-21]
 [3.7578473e-06 9.9365592e-01 6.3403151e-03]
 [1.0000000e+00 9.3004715e-10 6.6308126e-23]
 [9.9997771e-01 2.2293907e-05 3.9376556e-18]
 [2.5845482e-08 8.2792910e-03 9.9172068e-01]
 [6.3721098e-05 9.9841881e-01 1.5175065e-03]]
Przewidziane klasy:
[0 2 1 1 0 1 0 0 2 1]


Model sieci neuronowej dobrze rozpoznaje gatunki kwiatów ze zbioru iris jest to jednak mały zbiór ale prosta sieć z jedną warstwą ukrytą wystarcza do skutecznej klasyfikacji.